# 02 — Intent Classification: Baselines + LLM Classifier
Uses outputs from notebook 01. Set your LLM provider in the CONFIG cell (Groq or Gemini free tiers work fine).

In [ ]:
# --- CONFIG ---
OUTPUT_DIR = "data/processed"
CONFIG_DIR = "config"
MODELS_DIR = "models"
LLM_PROVIDER = "groq"   # "groq" or "gemini"
LLM_MODEL = "llama-3.3-70b-versatile"   # groq model name, or e.g. "gemini-2.0-flash" for gemini
LABEL_SAMPLE_SIZE = 800
RANDOM_STATE = 3


In [ ]:
import os, time
import pandas as pd, yaml
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import pickle

os.makedirs(MODELS_DIR, exist_ok=True)

pairs_df = pd.read_csv(os.path.join(OUTPUT_DIR, "cleaned_pairs.csv"))
with open(os.path.join(CONFIG_DIR, "intents.yaml")) as f:
    intents = yaml.safe_load(f)
print(len(pairs_df), "pairs;", len(intents), "intents")


## LLM client (swap provider here, rest of the notebook doesn't change)

In [ ]:
def make_llm_call():
    if LLM_PROVIDER == "groq":
        from groq import Groq
        client = Groq(api_key=os.environ["GROQ_API_KEY"])
        def call(prompt, max_tokens=20):
            resp = client.chat.completions.create(
                model=LLM_MODEL, max_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}]
            )
            return resp.choices[0].message.content.strip()
        return call
    elif LLM_PROVIDER == "gemini":
        import google.generativeai as genai
        genai.configure(api_key=os.environ["GEMINI_API_KEY"])
        model = genai.GenerativeModel(LLM_MODEL)
        def call(prompt, max_tokens=20):
            resp = model.generate_content(prompt)
            return resp.text.strip()
        return call
    else:
        raise ValueError(f"Unknown provider {LLM_PROVIDER}")

llm_call = make_llm_call()
print(llm_call("Say OK if you can read this."))


In [ ]:
def classify_intent_llm(text, intents):
    intent_list = "\n".join(f"- {k}: {v}" for k, v in intents.items())
    prompt = f'''Classify this customer support message into exactly one intent from this list:
{intent_list}

Message: "{text}"

Respond with ONLY the intent key, nothing else.'''
    raw = llm_call(prompt, max_tokens=20)
    raw = raw.strip().lower().replace(' ', '_')
    return raw if raw in intents else "unmatched"


## Step 1 — Label a sample with the LLM (silver-standard labels)
Note for the decision log / report: these labels come from the same LLM used elsewhere, so this
is not an independent ground truth — only the hand-labeled golden set (notebook 04) is.

In [ ]:
sample = pairs_df.sample(n=min(LABEL_SAMPLE_SIZE, len(pairs_df)), random_state=RANDOM_STATE).reset_index(drop=True)

llm_intents = []
for i, t in enumerate(sample['customer_text_clean']):
    llm_intents.append(classify_intent_llm(t, intents))
    if i % 50 == 0:
        print(i, "/", len(sample))
sample['llm_intent'] = llm_intents

print(sample['llm_intent'].value_counts())
sample = sample[sample['llm_intent'] != 'unmatched'].reset_index(drop=True)
print("After dropping unmatched:", len(sample))


## Step 2 — Baseline 1: trivial (majority class)

In [ ]:
majority_class = sample['llm_intent'].mode()[0]
baseline1_preds = [majority_class] * len(sample)
print("Majority baseline accuracy:", accuracy_score(sample['llm_intent'], baseline1_preds))


## Step 3 — Baseline 2: TF-IDF + Logistic Regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    sample['customer_text_clean'], sample['llm_intent'],
    test_size=0.2, random_state=1, stratify=sample['llm_intent']
)

vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)
tfidf_preds = clf.predict(X_test_vec)

tfidf_acc = accuracy_score(y_test, tfidf_preds)
tfidf_f1 = f1_score(y_test, tfidf_preds, average='macro')
print("TF-IDF+LR accuracy:", tfidf_acc, "| macro F1:", tfidf_f1)

with open(os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl"), "wb") as f:
    pickle.dump(vectorizer, f)
with open(os.path.join(MODELS_DIR, "logreg_clf.pkl"), "wb") as f:
    pickle.dump(clf, f)


## Step 4 — Your real approach: LLM few-shot classifier, evaluated on the same held-out split

In [ ]:
test_df = pd.DataFrame({'text': X_test.values, 'true_intent': y_test.values})
test_df['llm_pred'] = test_df['text'].apply(lambda t: classify_intent_llm(t, intents))

llm_acc = accuracy_score(test_df['true_intent'], test_df['llm_pred'])
llm_f1 = f1_score(test_df['true_intent'], test_df['llm_pred'], average='macro', zero_division=0)
print("LLM classifier accuracy:", llm_acc, "| macro F1:", llm_f1)


## Step 5 — Comparison table

In [ ]:
comparison = pd.DataFrame([
    {"method": "Majority (trivial)", "accuracy": accuracy_score(sample['llm_intent'], baseline1_preds), "macro_f1": None},
    {"method": "TF-IDF + LogReg (simple)", "accuracy": tfidf_acc, "macro_f1": tfidf_f1},
    {"method": "LLM few-shot (proposed)", "accuracy": llm_acc, "macro_f1": llm_f1},
])
comparison.to_csv(os.path.join(OUTPUT_DIR, "classifier_comparison.csv"), index=False)
comparison


**Limitation to note in the report's "misleading headline number" section:** the LLM classifier's
labels were generated by the same prompt family used to create the training/reference labels in
Step 1 — this comparison is mainly useful for checking whether TF-IDF is "good enough" as a cheap
baseline, not as a trustworthy absolute accuracy number. The golden set in notebook 04 is what
actually validates final performance.